# View generated showers

Reads the files written by `python -m sparseshower.generate` (or `jobs/generate.sub`)
and plots them. Needs only numpy and matplotlib.

Each `shower_NNNNN.npz` lists the lit 2 cm voxels: `xyz` [m], `nphotons`, `ijk`, `box_m`.
Set `SHOWER_DIR` below and run all cells.

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

SHOWER_DIR = Path("../showers")   # folder with shower_*.npz  <-- change this
N_DETAIL = 3                      # showers drawn in full detail
MAX_POINTS_3D = 20000             # brightest voxels kept in the 3D views
CMAP = "Blues"

files = sorted(SHOWER_DIR.glob("shower_*.npz"))
print(f"{len(files)} showers in {SHOWER_DIR.resolve()}")
if (SHOWER_DIR / "run.json").exists():
    print(json.loads((SHOWER_DIR / "run.json").read_text()))

## Summary

In [ ]:
def load(path):
    s = np.load(path)
    return dict(name=path.stem, xyz=s["xyz"], q=s["nphotons"], ijk=s["ijk"], box=s["box_m"])

showers = [load(f) for f in files]
print(f"{'file':<14}{'voxels':>9}{'photons':>12}{'max/voxel':>11}   extent x, y, z [m]")
for s in showers:
    ext = s["xyz"].max(0) - s["xyz"].min(0)
    print(f"{s['name']:<14}{len(s['q']):>9}{s['q'].sum():>12.3g}{s['q'].max():>11.3g}   "
          + ", ".join(f"{e:.2f}" for e in ext))

## One shower in detail

Top: photon-weighted projections on the three planes (log colour scale, one bin = one voxel).
Bottom: 3D view of the brightest voxels, photons per voxel, and the profile along the
shower axis (the photon-weighted principal axis, oriented so its largest component is
positive, measured from the first lit voxel).

In [ ]:
def projection(ax, s, a, b, labels="xyz"):
    box, n = s["box"], 192
    h, _, _ = np.histogram2d(s["xyz"][:, a], s["xyz"][:, b], bins=n,
                             range=[box[a], box[b]], weights=s["q"])
    h = np.ma.masked_equal(h.T, 0)
    im = ax.imshow(h, origin="lower", extent=[*box[a], *box[b]], cmap=CMAP,
                   norm=LogNorm(vmin=max(h.min(), 1e-3), vmax=h.max()), interpolation="nearest")
    ax.set_xlabel(f"{labels[a]} [m]")
    ax.set_ylabel(f"{labels[b]} [m]")
    return im


def axis_profile(s):
    w = s["q"] / s["q"].sum()
    centre = (s["xyz"] * w[:, None]).sum(0)
    d = s["xyz"] - centre
    cov = (d * w[:, None]).T @ d
    axis = np.linalg.eigh(cov)[1][:, -1]
    axis = axis * np.sign(axis[np.abs(axis).argmax()])   # largest component positive
    t = d @ axis
    return t - t.min(), axis


def scatter3d(ax, s):
    keep = np.argsort(s["q"])[::-1][:MAX_POINTS_3D]
    p, q = s["xyz"][keep], s["q"][keep]
    sc = ax.scatter(p[:, 0], p[:, 1], p[:, 2], c=np.log10(q), s=1.5, cmap=CMAP,
                    vmin=np.log10(q).min() - 0.3 * np.ptp(np.log10(q)), linewidths=0)
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.set_zlabel("z [m]")
    return sc


def plot_shower(s):
    fig = plt.figure(figsize=(15, 9.5))
    for k, (a, b) in enumerate([(0, 1), (0, 2), (1, 2)]):
        ax = fig.add_subplot(2, 3, k + 1)
        im = projection(ax, s, a, b)
        fig.colorbar(im, ax=ax, label="photons per voxel column", shrink=0.85)
    ax = fig.add_subplot(2, 3, 4, projection="3d")
    sc = scatter3d(ax, s)
    fig.colorbar(sc, ax=ax, label="log10 photons per voxel", shrink=0.6, pad=0.15)
    ax = fig.add_subplot(2, 3, 5)
    ax.hist(np.log10(s["q"]), bins=60, color="#2a6fb0")
    ax.set_xlabel("log10 photons per voxel")
    ax.set_ylabel("voxels")
    ax = fig.add_subplot(2, 3, 6)
    t, axis = axis_profile(s)
    h, e = np.histogram(t, bins=100, weights=s["q"])
    ax.step(e[:-1], h, where="post", color="#2a6fb0", lw=2)
    ax.set_yscale("log")
    ax.set_xlabel("distance along shower axis [m]")
    ax.set_ylabel("photons per bin")
    ax.set_title(f"axis = ({axis[0]:+.2f}, {axis[1]:+.2f}, {axis[2]:+.2f})", fontsize=9)
    fig.suptitle(f"{s['name']}: {len(s['q'])} voxels, {s['q'].sum():.3g} photons")
    fig.tight_layout()
    plt.show()


for s in showers[:N_DETAIL]:
    plot_shower(s)

## All showers side by side (x-z projection)

In [ ]:
def gallery(showers, a=0, b=2, ncol=5):
    nrow = int(np.ceil(len(showers) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.2 * ncol, 3.2 * nrow), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, s in zip(axes.flat, showers):
        ax.axis("on")
        projection(ax, s, a, b)
        ax.set_title(f"{s['name']}\n{s['q'].sum():.3g} photons", fontsize=8)
        ax.tick_params(labelsize=7)
    fig.tight_layout()
    plt.show()


gallery(showers[:40])

## Interactive 3D (optional)

Writes `<SHOWER_DIR>/<name>_3d.html` (open it in a browser; rotate and zoom with the mouse)
and shows it below. Uses plotly.js from a CDN, so the browser needs internet access.

In [ ]:
from IPython.display import HTML, display

HTML_TEMPLATE = """<!doctype html><html><head><meta charset="utf-8"><title>{name}</title>
<script src="https://cdn.jsdelivr.net/npm/plotly.js-dist-min@2.35.2/plotly.min.js"></script></head>
<body style="margin:0"><div id="{div}" style="width:100%;height:640px"></div><script>
var d = {data};
Plotly.newPlot("{div}", [{{type: "scatter3d", mode: "markers", x: d.x, y: d.y, z: d.z, text: d.t,
  hovertemplate: "x %{{x:.3f}} m<br>y %{{y:.3f}} m<br>z %{{z:.3f}} m<br>%{{text}} photons<extra></extra>",
  marker: {{size: 1.8, color: d.c, colorscale: "Blues", cmin: d.cmin, cmax: d.cmax,
           colorbar: {{title: "log10 photons"}}}}}}],
  {{title: "{name}: {nvox} voxels, {total} photons", margin: {{l: 0, r: 0, t: 40, b: 0}},
   scene: {{aspectmode: "data", xaxis: {{title: "x [m]"}}, yaxis: {{title: "y [m]"}}, zaxis: {{title: "z [m]"}}}}}});
</script></body></html>"""


def write_html(s, out_dir=SHOWER_DIR):
    keep = np.argsort(s["q"])[::-1][:MAX_POINTS_3D]
    p, q = s["xyz"][keep], s["q"][keep]
    c = np.log10(q)
    data = dict(x=np.round(p[:, 0], 4).tolist(), y=np.round(p[:, 1], 4).tolist(),
                z=np.round(p[:, 2], 4).tolist(), c=np.round(c, 3).tolist(),
                t=[f"{v:.3g}" for v in q], cmin=float(c.min() - 0.3 * np.ptp(c)), cmax=float(c.max()))
    html = HTML_TEMPLATE.format(name=s["name"], div=f"plot_{s['name']}", data=json.dumps(data),
                                nvox=len(s["q"]), total=f"{s['q'].sum():.3g}")
    path = Path(out_dir) / f"{s['name']}_3d.html"
    path.write_text(html)
    return path, html


for s in showers[:N_DETAIL]:
    path, html = write_html(s)
    print("wrote", path)
    display(HTML(html))